# Home Credit Default Risk — Preprocessing Pipeline

Clean pipeline notebook. See `home_credit_default_risk.ipynb` for EDA and the reasoning behind each step.

**Steps:**
1. Drop SK_ID columns (identifiers)
2. Drop low-variance columns (>95% dominant value)
3. Drop redundant aggregations (keep `_MEDI`, drop `_AVG` and `_MODE`)
4. Drop noisy columns (`AMT_REQ_CREDIT_BUREAU_*` — no meaningful signal)
5. Bucket high-cardinality categoricals by default rate, OHE the rest, drop >30% NaN
6. Drop highly correlated feature pairs (keep the one with stronger target correlation)
7. Align train/test columns (fill rare train-only dummy columns with 0 in test)

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

# Locate repo root — works whether running locally or on Binder/Colab
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.git').exists()), Path('../../'))
sys.path.insert(0, str(_root / 'src'))

from data_loader import ensure_data, RAW_DIR, PROCESSED_DIR
ensure_data()


def load_and_save_processed(raw_path, processed_path):
    """Load raw CSV and save a copy to the processed folder."""
    os.makedirs(os.path.dirname(processed_path), exist_ok=True)
    df = pd.read_csv(raw_path)
    df.to_csv(processed_path, index=False)
    print(f"Saved: {processed_path}  shape={df.shape}")
    return df


df_train_raw = load_and_save_processed(
    RAW_DIR / 'application_train.csv',
    PROCESSED_DIR / 'application_train.csv',
)
df_test_raw = load_and_save_processed(
    RAW_DIR / 'application_test.csv',
    PROCESSED_DIR / 'application_test.csv',
)

y_train = df_train_raw['TARGET']
X_train = df_train_raw.drop(columns=['TARGET'])

print(f"\nX_train: {X_train.shape} | df_test_raw: {df_test_raw.shape}")

In [ ]:
class DropIDColumns(BaseEstimator, TransformerMixin):
    """Drop SK_ID columns — identifiers with no predictive value."""
    def fit(self, X, y=None): return self
    def transform(self, X):
        cols = [col for col in X.columns if 'SK_ID' in col]
        return X.drop(columns=cols)


class DropLowVariance(BaseEstimator, TransformerMixin):
    """Drop columns where one value dominates more than 95% of rows."""
    def fit(self, X, y=None):
        self.cols_to_drop_ = [
            col for col in X.columns
            if X[col].value_counts(normalize=True).iloc[0] > 0.95
        ]
        return self
    def transform(self, X):
        return X.drop(columns=[c for c in self.cols_to_drop_ if c in X.columns])


class DropRedundantAggregations(BaseEstimator, TransformerMixin):
    """Drop _AVG and _MODE building columns — keep _MEDI (median is more robust)."""
    def fit(self, X, y=None): return self
    def transform(self, X):
        cols = [col for col in X.columns if '_AVG' in col or '_MODE' in col]
        return X.drop(columns=[c for c in cols if c in X.columns])


class DropNoisyColumns(BaseEstimator, TransformerMixin):
    """Drop AMT_REQ_CREDIT_BUREAU columns — shown to be noise with no meaningful signal."""
    def fit(self, X, y=None): return self
    def transform(self, X):
        cols = [col for col in X.columns if 'AMT_REQ_CREDIT_BUREAU' in col]
        return X.drop(columns=[c for c in cols if c in X.columns])


class BucketAndEncodeObjects(BaseEstimator, TransformerMixin):
    """
    Handle object (categorical) columns:
    - If >30% NaN  → drop column
    - If >10 unique values → bucketize by target default rate into 0/1/2
    - If <=10 unique values → one-hot encode
    """
    def fit(self, X, y):
        self.cols_to_drop_ = []
        self.bucket_maps_ = {}
        self.ohe_cols_ = []
        obj_cols = X.select_dtypes(include='object').columns
        for col in obj_cols:
            missing_rate = X[col].isnull().mean()
            if missing_rate > 0.30:
                self.cols_to_drop_.append(col)
                continue
            n_unique = X[col].nunique()
            if n_unique > 10:
                filled = X[col].fillna('MISSING')
                rates = filled.groupby(filled).apply(lambda s: y[s.index].mean())
                thresholds = rates.quantile([1/3, 2/3]).values
                bucket_map = {}
                for cat, rate in rates.items():
                    if rate <= thresholds[0]:   bucket_map[cat] = 0
                    elif rate <= thresholds[1]: bucket_map[cat] = 1
                    else:                       bucket_map[cat] = 2
                self.bucket_maps_[col] = bucket_map
            else:
                self.ohe_cols_.append(col)
        return self

    def transform(self, X):
        X = X.copy()
        X = X.drop(columns=[c for c in self.cols_to_drop_ if c in X.columns])
        for col, bucket_map in self.bucket_maps_.items():
            if col in X.columns:
                X[col] = X[col].fillna('MISSING').map(bucket_map).fillna(1).astype(int)
        ohe_cols = [c for c in self.ohe_cols_ if c in X.columns]
        if ohe_cols:
            X = pd.get_dummies(X, columns=ohe_cols, drop_first=True)
        return X


class DropHighlyCorrelated(BaseEstimator, TransformerMixin):
    """For each highly correlated pair (>threshold), drop the one with weaker target correlation."""
    def __init__(self, threshold=0.8):
        self.threshold = threshold

    def fit(self, X, y):
        df = X.copy()
        df['_TARGET'] = y.values
        corr_matrix = df.corr().abs()
        target_corr = corr_matrix['_TARGET']
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        self.cols_to_drop_ = set()
        for col in upper.columns:
            for row in upper.index:
                val = upper.loc[row, col]
                if pd.notna(val) and val > self.threshold and col != '_TARGET' and row != '_TARGET':
                    drop = col if target_corr.get(col, 0) <= target_corr.get(row, 0) else row
                    self.cols_to_drop_.add(drop)
        self.cols_to_drop_.discard('_TARGET')
        return self

    def transform(self, X):
        return X.drop(columns=[c for c in self.cols_to_drop_ if c in X.columns])


class AlignColumns(BaseEstimator, TransformerMixin):
    """Ensure test data has exactly the same columns as train after pipeline."""
    def fit(self, X, y=None):
        self.columns_ = X.columns.tolist()
        return self
    def transform(self, X):
        return X.reindex(columns=self.columns_, fill_value=0)

In [ ]:
preprocessing = Pipeline([
    ('drop_ids',          DropIDColumns()),
    ('drop_low_variance', DropLowVariance()),
    ('drop_aggregations', DropRedundantAggregations()),
    ('drop_noisy',        DropNoisyColumns()),
    ('encode_objects',    BucketAndEncodeObjects()),
    ('drop_correlated',   DropHighlyCorrelated(threshold=0.8)),
    ('align_columns',     AlignColumns()),
])

X_processed      = preprocessing.fit_transform(X_train, y_train)
X_test_processed = preprocessing.transform(df_test_raw)

print(f"Train processed: {X_processed.shape}")
print(f"Test processed:  {X_test_processed.shape}")